In [2]:
# =========================
# 1) Install dependencies
# =========================
!pip -q install transformers accelerate sentence-transformers faiss-cpu gradio pypdf python-docx


In [3]:
# =========================
# 2) Imports
# =========================
import os
import re
import math
import torch
import faiss
import gradio as gr

from google.colab import files
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
# Upload docx

In [5]:
print("Upload your PDF / TXT / DOCX files")
uploaded = files.upload()

Upload your PDF / TXT / DOCX files


Saving About the Program AI bot doc.docx to About the Program AI bot doc.docx


In [6]:
# =========================
# 4) Read files
# =========================
def read_pdf(path):
    text = []
    reader = PdfReader(path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

def read_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

all_docs = []

for filename in uploaded.keys():
    path = filename
    ext = filename.lower().split(".")[-1]

    try:
        if ext == "pdf":
            text = read_pdf(path)
        elif ext == "txt":
            text = read_txt(path)
        elif ext == "docx":
            text = read_docx(path)
        else:
            print(f"Skipping unsupported file: {filename}")
            continue

        if text.strip():
            all_docs.append({
                "source": filename,
                "text": text
            })
            print(f"Loaded: {filename} ({len(text)} chars)")
        else:
            print(f"No text found in: {filename}")

    except Exception as e:
        print(f"Error reading {filename}: {e}")

if not all_docs:
    raise ValueError("No readable text found in uploaded documents.")


Loaded: About the Program AI bot doc.docx (5127 chars)


In [7]:
# =========================
# 5) Chunk the documents
# =========================
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, chunk_size=700, overlap=120):
    text = clean_text(text)
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

chunks = []
for doc in all_docs:
    doc_chunks = chunk_text(doc["text"])
    for i, ch in enumerate(doc_chunks):
        chunks.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": ch
        })

print(f"Total chunks: {len(chunks)}")


Total chunks: 2


In [8]:
# =========================
# 6) Build embeddings + FAISS index
# =========================
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

chunk_texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity with normalized embeddings
index.add(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# now load the local model

In [11]:
llm_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
# =========================
# 8) Retrieval function
# =========================
def retrieve_context(query, top_k=4):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = chunks[idx].copy()
        item["score"] = float(score)
        results.append(item)
    return results

In [13]:
#prompt + ans generation

In [15]:
def build_prompt(question, retrieved):
    context_blocks = []
    for i, item in enumerate(retrieved, 1):
        context_blocks.append(
            f"[Document {i}] Source: {item['source']}\n{item['text']}"
        )

    context_text = "\n\n".join(context_blocks)

    prompt = f"""
You are a strict document question-answering assistant.

Rules:
1. Answer ONLY from the provided context.
2. If the answer is not clearly in the context, say exactly:
   "I don't know based on the uploaded documents."
3. Do not use outside knowledge.
4. Keep the answer clear and direct.
5. If possible, mention the source file name.
6. Make sure user not leave
7. you can suggest question from your end

Context:
{context_text}

Question:
{question}

Answer:
""".strip()

    return prompt

def generate_answer(question, max_new_tokens=220):
    retrieved = retrieve_context(question, top_k=4)

    # Optional: if retrieval confidence is weak, refuse early
    best_score = retrieved[0]["score"] if retrieved else 0.0
    if best_score < 0.25:
        return "I don't know based on the uploaded documents."

    prompt = build_prompt(question, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Answer:" in full_text:
        answer = full_text.split("Answer:", 1)[-1].strip()
    else:
        answer = full_text.strip()

    # Safety filter to keep it docs-only
    weak_phrases = [
        "as an ai",
        "general knowledge",
        "typically",
        "usually",
    ]
    if any(p in answer.lower() for p in weak_phrases):
        return "I don't know based on the uploaded documents."

    if len(answer) < 3:
        return "I don't know based on the uploaded documents."

    return answer



In [17]:
# =========================
# 10) Chat function for Gradio
# =========================
def chat_fn(message, history):
    answer = generate_answer(message)

    # show top sources underneath
    retrieved = retrieve_context(message, top_k=3)
    if retrieved:
        srcs = []
        for r in retrieved:
            srcs.append(f"- {r['source']} (score: {r['score']:.3f})")
        sources_text = "\n\nSources used:\n" + "\n".join(srcs)
        answer = answer + sources_text

    return answer


In [18]:
# =========================
# 11) Launch chatbot UI
# =========================
demo = gr.ChatInterface(
    fn=chat_fn,
    title="Docs-Only Chatbot",
    description="Ask questions about your uploaded documents only."
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75870f66399bbc0d71.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
